# 05 Makemore Backprop Ninja Exercises

This notebook works through the three manual-backprop exercises in the most explicit way possible.

The goal is not to memorize the answers. The goal is to learn how to derive each backward line from the forward line that created the tensor.

The rule we will use throughout:

```text
forward made a value
backward asks: how does loss change when that value changes?
```

For every operation, keep asking four shape questions:

```text
1. Did forward select only some entries?        -> backward writes zeros elsewhere.
2. Did forward broadcast one value many times? -> backward sums those many uses.
3. Did forward sum many values into one?       -> backward broadcasts/copies back.
4. Did one tensor feed multiple forward paths? -> backward adds the path gradients.
```

We will compare every manual gradient to PyTorch autograd. PyTorch is the answer checker, not the teacher. The teaching is in deriving the gradient before checking it.


## Setup

The model is the same small Makemore-style network used in the backprop exercise:

```text
characters -> embeddings -> flatten -> linear -> BatchNorm -> tanh -> linear -> softmax loss
```

The important tensors:

```text
Xb: integer input character IDs
Yb: integer target character IDs
C: embedding table
W1, b1: first linear layer
bngain, bnbias: BatchNorm scale and shift
W2, b2: output layer
```

The notebook deliberately keeps the forward pass decomposed into tiny steps so each backward step has one rule to apply.


In [1]:
from pathlib import Path
import random

import torch
import torch.nn.functional as F

# Make the path work whether Jupyter's current directory is the repo root
# or the notebooks/ directory.
ROOT = Path.cwd()
if not (ROOT / "data" / "names.txt").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "names.txt"
assert DATA_PATH.exists(), f"Could not find {DATA_PATH}"

words = DATA_PATH.read_text().splitlines()
chars = sorted(set("".join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
block_size = 3

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

Xtr.shape, Ytr.shape, vocab_size


(torch.Size([182625, 3]), torch.Size([182625]), 27)

In [2]:
# Model sizes.
n_embd = 10
n_hidden = 64
n = 32  # batch size used by the manual-backprop exercise

g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g).requires_grad_()
W1 = (torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5).requires_grad_()
b1 = (torch.randn(n_hidden, generator=g) * 0.1).requires_grad_()
W2 = (torch.randn((n_hidden, vocab_size), generator=g) * 0.1).requires_grad_()
b2 = (torch.randn(vocab_size, generator=g) * 0.1).requires_grad_()
bngain = (torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0).requires_grad_()
bnbias = (torch.randn((1, n_hidden), generator=g) * 0.1).requires_grad_()

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters), "parameters")


4137 parameters


## Forward Pass, Kept Intentionally Verbose

Do not simplify this yet. Each line below becomes one or more backward lines.

The softmax is written by hand:

```text
logits -> subtract row max -> exp -> sum -> inverse -> probabilities -> log -> negative mean
```

The BatchNorm is also written by hand:

```text
hprebn -> mean -> subtract mean -> square -> variance -> inverse std -> normalize -> gain/bias
```

This is why the manual backward pass is long: we are refusing to hide any operation.


In [3]:
# Sample one minibatch.
ix = torch.randint(0, Xtr.shape[0], (n,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

# Clear old parameter gradients if this cell is re-run.
for p in parameters:
    p.grad = None

# Forward pass.
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

hprebn = embcat @ W1 + b1
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1 / (n - 1) * bndiff2.sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)

logits = h @ W2 + b2
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# Retain gradients for non-leaf tensors so cmp() can inspect them.
for t in [
    emb, embcat, hprebn, bnmeani, bndiff, bndiff2, bnvar, bnvar_inv,
    bnraw, hpreact, h, logits, logit_maxes, norm_logits, counts,
    counts_sum, counts_sum_inv, probs, logprobs,
]:
    t.retain_grad()

loss.backward(retain_graph=True)
print("loss:", loss.item())
print("Xb shape:", Xb.shape, "Yb shape:", Yb.shape)
print("logits shape:", logits.shape)


loss: 3.5571470260620117
Xb shape: torch.Size([32, 3]) Yb shape: torch.Size([32])
logits shape: torch.Size([32, 27])


In [4]:
def cmp(name, manual, tensor, atol=1e-7):
    """Compare a manual gradient against PyTorch's autograd gradient."""
    assert tensor.grad is not None, f"{name}: tensor.grad is None; did you call retain_grad/backward?"
    exact = torch.equal(manual, tensor.grad)
    approx = torch.allclose(manual, tensor.grad, atol=atol, rtol=atol)
    maxdiff = (manual - tensor.grad).abs().max().item()
    print(f"{name:16s} | exact: {str(exact):5s} | approximate: {str(approx):5s} | maxdiff: {maxdiff:.3e}")


# Exercise 1: Backprop Through Every Forward Variable

We now derive every backward line in the same order as the forward pass, but reversed.

Start at the loss:

```text
loss = -logprobs[range(n), Yb].mean()
```

`loss` is a scalar. By convention:

```text
dloss/dloss = 1
```

We usually do not write `dloss = 1` in code because every following line already computes gradients with respect to the final loss.


## 1. `logprobs`: only selected entries affect the loss

Forward:

```python
loss = -logprobs[range(n), Yb].mean()
```

What is `logprobs`?

```text
logprobs.shape = [n, vocab_size]
```

Each row is one training example. Each column is one possible next character. `logprobs[i, c]` is the log probability assigned to class `c` for example `i`.

What is `Yb`?

```text
Yb.shape = [n]
```

`Yb[i]` is the correct class index for example `i`.

What is `range(n)`?

```text
range(n) = [0, 1, 2, ..., n-1]
```

So:

```python
logprobs[range(n), Yb]
```

means:

```text
pick one cell from each row:
row 0, correct column Yb[0]
row 1, correct column Yb[1]
...
row n-1, correct column Yb[n-1]
```

Only those selected log-probabilities are used by the loss. Every unselected class has zero direct gradient at this step.

Why `torch.zeros_like(logprobs)`?

The gradient of `logprobs` must have the same shape as `logprobs`, `[n, vocab_size]`. We start with zeros because most entries were not selected by the indexing operation.

Mean rule:

```text
mean of n selected values -> each selected value gets 1/n of the upstream gradient
```

Negative sign rule:

```text
loss = -selected.mean() -> selected gradient is -1/n
```


In [5]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n

cmp("logprobs", dlogprobs, logprobs)
print("selected entries per row:", list(zip(range(5), Yb[:5].tolist())))


logprobs         | exact: True  | approximate: True  | maxdiff: 0.000e+00
selected entries per row: [(0, 1), (1, 0), (2, 25), (3, 12), (4, 9)]


## 2. `probs`: derivative of log

Forward:

```python
logprobs = probs.log()
```

Local card:

```text
z = log(x)
dx = dz * (1 / x)
```

So:

```text
dprobs = dlogprobs * (1 / probs)
```

Intuition: log is steep near zero. If the model gives a tiny probability to the true class, `1 / probs` is large, so the gradient is large.


In [6]:
dprobs = (1.0 / probs) * dlogprobs

cmp("probs", dprobs, probs)


probs            | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 3. `counts_sum_inv`: multiplication with broadcasting

Forward:

```python
probs = counts * counts_sum_inv
```

Shapes:

```text
counts.shape         = [n, vocab_size]
counts_sum_inv.shape = [n, 1]
probs.shape          = [n, vocab_size]
```

Multiplication card:

```text
z = x * y
dx = dz * y
dy = dz * x
```

Direct gradient to `counts` from this multiply:

```text
dcounts_from_probs = dprobs * counts_sum_inv
```

But `counts_sum_inv` was broadcast across all classes in each row. One value `counts_sum_inv[i, 0]` was reused for every class `c` in row `i`.

Broadcast rule:

```text
broadcast in forward -> sum in backward
```

So the gradient for `counts_sum_inv` must add up all class contributions in each row:

```text
dcounts_sum_inv[i, 0] = sum_c counts[i, c] * dprobs[i, c]
```


In [7]:
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs

cmp("counts_sum_inv", dcounts_sum_inv, counts_sum_inv)
# We do not compare counts yet because counts has another path through counts_sum.


counts_sum_inv   | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 4. `counts_sum`: power rule

Forward:

```python
counts_sum_inv = counts_sum ** -1
```

Power card:

```text
z = x ** k
dx = dz * k * x ** (k - 1)
```

Here `k = -1`, so:

```text
d/dx x^-1 = -1 * x^-2
```

Therefore:

```text
dcounts_sum = dcounts_sum_inv * (-counts_sum ** -2)
```

Intuition: increasing the denominator lowers the inverse. That is why the gradient is negative.


In [8]:
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv

cmp("counts_sum", dcounts_sum, counts_sum)


counts_sum       | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 5. `counts`: sum backward broadcasts, and repeated use adds

Forward:

```python
counts_sum = counts.sum(1, keepdim=True)
```

Shapes:

```text
counts.shape     = [n, vocab_size]
counts_sum.shape = [n, 1]
```

Sum rule:

```text
sum in forward -> broadcast/copy in backward
```

For each row, many class values were added into one denominator. In backward, that one denominator gradient is copied back to every class in the row:

```text
dcounts_from_sum = ones_like(counts) * dcounts_sum
```

But `counts` already received a gradient from:

```python
probs = counts * counts_sum_inv
```

So `counts` has two forward paths:

```text
counts -> probs -> loss
counts -> counts_sum -> counts_sum_inv -> probs -> loss
```

Fork rule:

```text
one value used twice in forward -> add gradients in backward
```


In [9]:
dcounts += torch.ones_like(counts) * dcounts_sum

cmp("counts", dcounts, counts)


counts           | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 6. `norm_logits`: exp backward

Forward:

```python
counts = norm_logits.exp()
```

Exp card:

```text
z = exp(x)
dx = dz * exp(x)
```

Because `counts` already stores `exp(norm_logits)`, we can write:

```text
dnorm_logits = dcounts * counts
```


In [10]:
dnorm_logits = counts * dcounts

cmp("norm_logits", dnorm_logits, norm_logits)


norm_logits      | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 7. `logits` and `logit_maxes`: subtraction plus max routing

Forward:

```python
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
```

First handle the subtraction:

```text
z = x - y
dx = dz
dy = -dz
```

So direct contribution to `logits` is:

```text
dlogits_from_norm = dnorm_logits
```

`logit_maxes` was broadcast across all classes in a row before subtraction. Broadcast rule says we sum over the broadcasted class dimension:

```text
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
```

Now backprop through:

```python
logit_maxes = logits.max(1, keepdim=True).values
```

Max card:

```text
max routes gradient only to the winning input
```

For each row, only the class that had the maximum logit receives `dlogit_maxes`. We use `one_hot(argmax)` as a routing mask.

Important intuition: subtracting the row max is for numerical stability. It should not change the softmax probabilities. In backward, the gradient through this max path almost cancels because softmax gradients in each row sum to zero.


In [11]:
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

cmp("logit_maxes", dlogit_maxes, logit_maxes)
cmp("logits", dlogits, logits)
print("row sums of dlogits, first five:", dlogits[:5].sum(1))


logit_maxes      | exact: True  | approximate: True  | maxdiff: 0.000e+00
logits           | exact: True  | approximate: True  | maxdiff: 0.000e+00
row sums of dlogits, first five: tensor([ 9.3132e-10, -6.9849e-10, -2.3283e-10,  0.0000e+00, -6.9849e-10],
       grad_fn=<SumBackward1>)


## 8. Output layer: matrix multiply plus bias broadcast

Forward:

```python
logits = h @ W2 + b2
```

Shapes:

```text
h.shape       = [n, n_hidden]
W2.shape      = [n_hidden, vocab_size]
b2.shape      = [vocab_size]
logits.shape  = [n, vocab_size]
```

Matrix multiply card:

```text
Z = X @ W
dX = dZ @ W.T
dW = X.T @ dZ
```

Why the transposes? Shape reasoning:

```text
dh must be [n, n_hidden]
therefore dlogits [n, vocab_size] @ W2.T [vocab_size, n_hidden]

dW2 must be [n_hidden, vocab_size]
therefore h.T [n_hidden, n] @ dlogits [n, vocab_size]
```

Bias rule:

```text
b2 was broadcast across all n rows
broadcast in forward -> sum over rows in backward
```


In [12]:
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

cmp("h", dh, h)
cmp("W2", dW2, W2)
cmp("b2", db2, b2)


h                | exact: True  | approximate: True  | maxdiff: 0.000e+00
W2               | exact: True  | approximate: True  | maxdiff: 0.000e+00
b2               | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 9. `hpreact`: tanh backward

Forward:

```python
h = torch.tanh(hpreact)
```

Tanh card:

```text
z = tanh(x)
dx = dz * (1 - tanh(x)^2)
```

Since `h` already equals `tanh(hpreact)`, use:

```text
dhpreact = dh * (1 - h^2)
```

Intuition: when tanh saturates near `-1` or `+1`, `1 - h^2` is near zero. That kills gradient flow through the activation.


In [13]:
dhpreact = (1.0 - h ** 2) * dh

cmp("hpreact", dhpreact, hpreact)


hpreact          | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 10. BatchNorm scale and shift: `bngain`, `bnbias`, `bnraw`

Forward:

```python
hpreact = bngain * bnraw + bnbias
```

Shapes:

```text
hpreact.shape = [n, n_hidden]
bnraw.shape   = [n, n_hidden]
bngain.shape  = [1, n_hidden]
bnbias.shape  = [1, n_hidden]
```

First the addition:

```text
hpreact = something + bnbias
```

`bnbias` was broadcast over the batch rows, so:

```text
dbnbias = dhpreact.sum(0, keepdim=True)
```

Then the multiply:

```text
something = bngain * bnraw
```

Multiplication card:

```text
dbnraw = dhpreact * bngain
dbngain_big = dhpreact * bnraw
```

But `bngain` was broadcast over rows, so we sum over rows:

```text
dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
```


In [14]:
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)

cmp("bngain", dbngain, bngain)
cmp("bnraw", dbnraw, bnraw)
cmp("bnbias", dbnbias, bnbias)


bngain           | exact: True  | approximate: True  | maxdiff: 0.000e+00
bnraw            | exact: True  | approximate: True  | maxdiff: 0.000e+00
bnbias           | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 11. `bnraw`: normalized value times inverse standard deviation

Forward:

```python
bnraw = bndiff * bnvar_inv
```

Shapes:

```text
bndiff.shape    = [n, n_hidden]
bnvar_inv.shape = [1, n_hidden]
bnraw.shape     = [n, n_hidden]
```

Multiplication card:

```text
dbndiff_from_bnraw = dbnraw * bnvar_inv
dbnvar_inv_big     = dbnraw * bndiff
```

`bnvar_inv` was broadcast down the batch rows, so we add its row contributions:

```text
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
```


In [15]:
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

cmp("bnvar_inv", dbnvar_inv, bnvar_inv)
# We do not compare bndiff yet because bndiff has another path through bndiff2.


bnvar_inv        | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 12. `bnvar`: inverse square root power rule

Forward:

```python
bnvar_inv = (bnvar + 1e-5) ** -0.5
```

Power card:

```text
z = x ** k
dx = dz * k * x ** (k - 1)
```

Here:

```text
x = bnvar + 1e-5
k = -0.5
```

So:

```text
dbnvar = dbnvar_inv * (-0.5) * (bnvar + 1e-5) ** -1.5
```

The `1e-5` is a constant, so it does not receive a gradient. It is there to prevent division by zero when variance is tiny.


In [16]:
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv

cmp("bnvar", dbnvar, bnvar)


bnvar            | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 13. `bndiff2`: variance is a scaled sum

Forward:

```python
bnvar = 1 / (n - 1) * bndiff2.sum(0, keepdim=True)
```

This has two simple operations:

```text
sum over batch rows
multiply by 1/(n-1)
```

Sum rule:

```text
sum in forward -> broadcast/copy in backward
```

Scale rule:

```text
z = a * x
dx = a * dz
```

So every row gets the same `dbnvar`, scaled by `1/(n-1)`:

```text
dbndiff2 = ones_like(bndiff2) * dbnvar / (n - 1)
```

The `(n - 1)` is Bessel's correction: this exercise uses unbiased variance.


In [17]:
dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar

cmp("bndiff2", dbndiff2, bndiff2)


bndiff2          | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 14. `bndiff`: square backward, then add the two paths

Forward:

```python
bndiff2 = bndiff ** 2
```

Power card:

```text
z = x^2
dx = dz * 2x
```

So the variance path contributes:

```text
dbndiff_from_var = dbndiff2 * 2 * bndiff
```

But `bndiff` was already used in:

```python
bnraw = bndiff * bnvar_inv
```

So `bndiff` has two paths:

```text
bndiff -> bnraw -> loss
bndiff -> bndiff2 -> bnvar -> bnvar_inv -> bnraw -> loss
```

Fork rule:

```text
one value used twice in forward -> add gradients in backward
```


In [18]:
dbndiff += (2 * bndiff) * dbndiff2

cmp("bndiff", dbndiff, bndiff)


bndiff           | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 15. `hprebn` and `bnmeani`: subtraction plus broadcasted mean

Forward:

```python
bndiff = hprebn - bnmeani
```

Subtraction card:

```text
z = x - y
dx = dz
dy = -dz
```

Direct contribution:

```text
dhprebn_from_bndiff = dbndiff
```

For `bnmeani`, remember the shape:

```text
bnmeani.shape = [1, n_hidden]
bndiff.shape  = [n, n_hidden]
```

`bnmeani` was broadcast down the rows. Broadcast rule says sum over the rows:

```text
dbnmeani = (-dbndiff).sum(0, keepdim=True)
```

Now backprop through the mean:

```python
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
```

Mean rule:

```text
each row receives 1/n of the upstream mean gradient
```

So:

```text
dhprebn_from_mean = ones_like(hprebn) * dbnmeani / n
```

Finally add the two paths into `hprebn`.


In [19]:
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0, keepdim=True)
dhprebn += (1.0 / n) * torch.ones_like(hprebn) * dbnmeani

cmp("bnmeani", dbnmeani, bnmeani)
cmp("hprebn", dhprebn, hprebn)


bnmeani          | exact: True  | approximate: True  | maxdiff: 0.000e+00
hprebn           | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 16. First linear layer: `embcat @ W1 + b1`

Forward:

```python
hprebn = embcat @ W1 + b1
```

Matrix multiply card:

```text
Z = X @ W
dX = dZ @ W.T
dW = X.T @ dZ
```

So:

```text
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
```

Bias rule:

```text
b1 was broadcast over n rows -> db1 = dhprebn.sum(0)
```

`db1` does not need `keepdim=True` here because `b1.shape` is `[n_hidden]`, not `[1, n_hidden]`.


In [20]:
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

cmp("embcat", dembcat, embcat)
cmp("W1", dW1, W1)
cmp("b1", db1, b1)


embcat           | exact: True  | approximate: True  | maxdiff: 0.000e+00
W1               | exact: True  | approximate: True  | maxdiff: 0.000e+00
b1               | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 17. `embcat`: view backward just reshapes the gradient

Forward:

```python
embcat = emb.view(emb.shape[0], -1)
```

A view changes shape but not the underlying values.

Forward shape:

```text
emb.shape    = [n, block_size, n_embd]
embcat.shape = [n, block_size * n_embd]
```

Backward rule:

```text
reshape/view in forward -> reshape gradient back in backward
```

So:

```text
demb = dembcat.view(emb.shape)
```


In [21]:
demb = dembcat.view(emb.shape)

cmp("emb", demb, emb)


emb              | exact: True  | approximate: True  | maxdiff: 0.000e+00


## 18. `C`: indexing backward is scatter-add

Forward:

```python
emb = C[Xb]
```

`C` is the embedding table:

```text
C.shape = [vocab_size, n_embd]
```

`Xb` contains integer row indices into `C`:

```text
Xb.shape = [n, block_size]
```

Forward gather:

```text
emb[k, j] = C[Xb[k, j]]
```

Backward asks:

```text
where should demb[k, j] go inside dC?
```

Answer:

```text
it goes back to row Xb[k, j]
```

Why start with zeros?

`dC` must have the same shape as `C`, but only rows that were selected in the batch receive gradient. Unselected embedding rows stay zero for this minibatch.

Why add instead of assign?

The same character index can appear many times in the batch. If row `3` is used ten times, all ten gradient vectors must accumulate into `dC[3]`.

Indexing rule:

```text
gather in forward -> scatter-add in backward
```


In [22]:
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]

cmp("C", dC, C)
print("example Xb row:", Xb[0].tolist())
print("those embedding gradients are added into rows:", Xb[0].tolist())


C                | exact: True  | approximate: True  | maxdiff: 0.000e+00
example Xb row: [0, 11, 25]
those embedding gradients are added into rows: [0, 11, 25]


## Exercise 1 Full Check

This cell compares every manual gradient from Exercise 1 against PyTorch autograd.

`exact=False` with `approximate=True` is fine for some tensors. Floating-point additions can happen in a slightly different order, especially around BatchNorm reductions.


In [23]:
checks = [
    ("logprobs", dlogprobs, logprobs),
    ("probs", dprobs, probs),
    ("counts_sum_inv", dcounts_sum_inv, counts_sum_inv),
    ("counts_sum", dcounts_sum, counts_sum),
    ("counts", dcounts, counts),
    ("norm_logits", dnorm_logits, norm_logits),
    ("logit_maxes", dlogit_maxes, logit_maxes),
    ("logits", dlogits, logits),
    ("h", dh, h),
    ("W2", dW2, W2),
    ("b2", db2, b2),
    ("hpreact", dhpreact, hpreact),
    ("bngain", dbngain, bngain),
    ("bnbias", dbnbias, bnbias),
    ("bnraw", dbnraw, bnraw),
    ("bnvar_inv", dbnvar_inv, bnvar_inv),
    ("bnvar", dbnvar, bnvar),
    ("bndiff2", dbndiff2, bndiff2),
    ("bndiff", dbndiff, bndiff),
    ("bnmeani", dbnmeani, bnmeani),
    ("hprebn", dhprebn, hprebn),
    ("embcat", dembcat, embcat),
    ("W1", dW1, W1),
    ("b1", db1, b1),
    ("emb", demb, emb),
    ("C", dC, C),
]

for name, manual, tensor in checks:
    cmp(name, manual, tensor, atol=1e-6)


logprobs         | exact: True  | approximate: True  | maxdiff: 0.000e+00
probs            | exact: True  | approximate: True  | maxdiff: 0.000e+00
counts_sum_inv   | exact: True  | approximate: True  | maxdiff: 0.000e+00
counts_sum       | exact: True  | approximate: True  | maxdiff: 0.000e+00
counts           | exact: True  | approximate: True  | maxdiff: 0.000e+00
norm_logits      | exact: True  | approximate: True  | maxdiff: 0.000e+00
logit_maxes      | exact: True  | approximate: True  | maxdiff: 0.000e+00
logits           | exact: True  | approximate: True  | maxdiff: 0.000e+00
h                | exact: True  | approximate: True  | maxdiff: 0.000e+00
W2               | exact: True  | approximate: True  | maxdiff: 0.000e+00
b2               | exact: True  | approximate: True  | maxdiff: 0.000e+00
hpreact          | exact: True  | approximate: True  | maxdiff: 0.000e+00
bngain           | exact: True  | approximate: True  | maxdiff: 0.000e+00
bnbias           | exact: True  | appr

# Exercise 2: Cross-Entropy Backward in One Step

Exercise 1 backpropagated through the softmax loss one tiny operation at a time:

```text
logits -> max -> subtract -> exp -> sum -> inverse -> multiply -> log -> index -> mean
```

Exercise 2 asks us to collapse that whole chain into one gradient for `logits`.

For one example, define:

$$
p_c = \frac{e^{\ell_c}}{\sum_k e^{\ell_k}}
$$

If the true class is `y`, the loss is:

$$
L = -\log p_y
$$

Now substitute softmax into the loss:

$$
L
= -\log \left(\frac{e^{\ell_y}}{\sum_k e^{\ell_k}}\right)
$$

Use the log rule:

$$
\log(a/b) = \log a - \log b
$$

So:

$$
L = -\ell_y + \log\sum_k e^{\ell_k}
$$

Now take the derivative with respect to one logit `l_c`.

There are two cases.

Case 1: `c` is not the true class.

```text
-l_y does not contain l_c, so that part gives 0.
log(sum exp(logits)) gives softmax probability p_c.
```

So:

$$
\frac{\partial L}{\partial \ell_c} = p_c
\quad\text{if } c \ne y
$$

Case 2: `c` is the true class.

```text
-l_y contributes -1.
log(sum exp(logits)) still contributes p_y.
```

So:

$$
\frac{\partial L}{\partial \ell_y} = p_y - 1
$$

Both cases can be written as one vector:

```text
dlogits = prediction - answer
```

where `answer` is the one-hot target vector.

Because the notebook loss averages over `n` examples, divide by `n`.


## Why `dlogits[range(n), Yb] -= 1`?

Start with:

```python
dlogits = F.softmax(logits, 1)
```

Now `dlogits` contains the prediction probabilities for every row and every class.

For every row `i`, only the true class `Yb[i]` needs the `-1` part:

```text
wrong classes:  p_c
true class:     p_y - 1
```

`range(n)` gives the row indices. `Yb` gives the correct column indices. Together:

```python
dlogits[range(n), Yb]
```

selects the true-class cell in every row.


In [24]:
dlogits_fast = F.softmax(logits, 1)
dlogits_fast[range(n), Yb] -= 1
dlogits_fast /= n

cmp("logits fast CE", dlogits_fast, logits, atol=1e-6)
print("first row target:", Yb[0].item())
print("first row gradient sums to:", dlogits_fast[0].sum().item())
print("first row true-class gradient:", dlogits_fast[0, Yb[0]].item())


logits fast CE   | exact: False | approximate: True  | maxdiff: 4.889e-09
first row target: 1
first row gradient sums to: 9.313225746154785e-10
first row true-class gradient: -0.030558263882994652


## Cross-Entropy Intuition

For one row:

```text
dlogits = probs - one_hot_answer
```

If the model is wrong and gives low probability to the correct class:

```text
p_correct - 1 is close to -1
```

Gradient descent subtracts the gradient, so a negative gradient makes the correct logit go up strongly.

If the model puts high probability on a wrong class:

```text
p_wrong is large and positive
```

Gradient descent subtracts it, so the wrong logit goes down strongly.

Every row sums to zero:

```text
sum(probs - one_hot) = 1 - 1 = 0
```

So the loss redistributes score: push the correct class up, pull the wrong classes down.


# Exercise 3: BatchNorm Backward in One Step

Exercise 1 backpropagated through BatchNorm one tiny operation at a time:

```text
hprebn -> mean -> subtract -> square -> variance -> inverse std -> normalize -> gain/bias
```

Exercise 3 asks us to collapse the BatchNorm part into one formula for:

```text
dhprebn
```

BatchNorm forward, for one feature column:

$$
\mu_B = \frac{1}{n}\sum_i x_i
$$

$$
\sigma_B^2 = \frac{1}{n-1}\sum_i (x_i - \mu_B)^2
$$

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
$$

$$
y_i = \gamma \hat{x}_i + \beta
$$

In the notebook names:

```text
x_i        -> one row of hprebn
mu_B       -> bnmeani
x_i-mu_B   -> bndiff
sigma_B^2  -> bnvar
1/std      -> bnvar_inv
xhat_i     -> bnraw
y_i        -> hpreact
```


## Why BatchNorm is harder than softmax

In a normal elementwise operation, row `i` often depends only on row `i`.

BatchNorm is different. One row affects the batch mean and batch variance, and those statistics are used by every other row.

So `x_i` affects the loss through three paths:

```text
1. direct path:   x_i changes its own normalized value
2. mean path:     x_i changes the batch mean used by all rows
3. variance path: x_i changes the batch variance used by all rows
```

That is why the final formula has three terms inside the bracket.


## The compact formula

For the unbiased variance used here, the compact backward is:

$$
\frac{\partial L}{\partial x_i}
=
\gamma\,\text{invstd}\,
\left[
\frac{\partial L}{\partial y_i}
-
\frac{1}{n}\sum_j \frac{\partial L}{\partial y_j}
-
\frac{n}{n-1}\hat{x}_i\frac{1}{n}\sum_j\frac{\partial L}{\partial y_j}\hat{x}_j
\right]
$$

Read the bracket as:

```text
raw upstream gradient
- column mean of upstream gradients
- variance/alignment correction
```

The last term removes the component of the gradient that would change the normalized vector's variance.

The `n/(n-1)` appears because this forward pass used unbiased variance, dividing by `n-1` instead of `n`.


## How the vector formula becomes tensor code

The formula above is for one feature column.

But `hprebn` is a matrix:

```text
hprebn.shape = [n, n_hidden]
```

BatchNorm applies the same formula independently to every hidden feature column.

That means:

```text
sum over j = sum over batch rows
keep one number per hidden feature
```

In PyTorch:

```python
dhpreact.sum(0, keepdim=True)
```

has shape:

```text
[1, n_hidden]
```

`keepdim=True` matters because it keeps the column shape so PyTorch can broadcast it back over the `n` rows.

The key tensor shapes:

```text
dhpreact:                    [n, n_hidden]
dhpreact.sum(0, keepdim=True): [1, n_hidden]
bnraw:                       [n, n_hidden]
(dhpreact * bnraw).sum(0, keepdim=True): [1, n_hidden]
```

So the vector formula is written once, and PyTorch applies it to every column in parallel.


In [25]:
print("dhpreact shape:", dhpreact.shape)
print("batch sum shape:", dhpreact.sum(0, keepdim=True).shape)
print("bnraw shape:", bnraw.shape)
print("alignment sum shape:", (dhpreact * bnraw).sum(0, keepdim=True).shape)
print("bngain shape:", bngain.shape)
print("bnvar_inv shape:", bnvar_inv.shape)


dhpreact shape: torch.Size([32, 64])
batch sum shape: torch.Size([1, 64])
bnraw shape: torch.Size([32, 64])
alignment sum shape: torch.Size([1, 64])
bngain shape: torch.Size([1, 64])
bnvar_inv shape: torch.Size([1, 64])


In [26]:
dhprebn_fast = bngain * bnvar_inv / n * (
    n * dhpreact
    - dhpreact.sum(0, keepdim=True)
    - (n / (n - 1)) * bnraw * (dhpreact * bnraw).sum(0, keepdim=True)
)

cmp("hprebn fast BN", dhprebn_fast, hprebn, atol=1e-6)
print("max difference from Exercise 1 dhprebn:", (dhprebn_fast - dhprebn).abs().max().item())


hprebn fast BN   | exact: False | approximate: True  | maxdiff: 9.313e-10
max difference from Exercise 1 dhprebn: 9.313225746154785e-10


## BatchNorm intuition from the formula

The compact formula:

```text
dx = scale * (direct - mean_correction - variance_correction)
```

Term by term:

```text
gamma * invstd
```

This is the local scale from the final BatchNorm output. If `gamma` is large, gradients through BatchNorm are scaled up. If the variance is tiny, `invstd` is large, so epsilon protects the division.

```text
dhpreact
```

This is what each row wants directly from upstream.

```text
- dhpreact.mean(0)
```

BatchNorm centers the batch. The output should not respond to shifting every input in a column by the same constant. This term removes the batch-average gradient.

```text
- bnraw * mean(dhpreact * bnraw) * n/(n-1)
```

BatchNorm also normalizes variance. This term removes the part of the gradient aligned with the normalized values themselves. The `n/(n-1)` adjusts for the unbiased variance denominator.


# Summary: Patterns Observed

The same small set of patterns solved the whole notebook.

| Forward pattern | Backward pattern | Where it appeared |
| --- | --- | --- |
| Select/index a few entries | Start with zeros, write gradients only to selected entries | `logprobs[range(n), Yb]`, `C[Xb]` |
| Broadcast one value many times | Sum over the broadcasted dimension | `counts_sum_inv`, `logit_maxes`, `bngain`, `bnbias`, `bnvar_inv`, `bnmeani`, biases |
| Sum many values into one | Broadcast/copy the upstream gradient back | `counts.sum(1)`, `hprebn.sum(0)`, `bndiff2.sum(0)` |
| One tensor used in two places | Add gradient contributions | `counts`, `bndiff`, `hprebn` |
| Elementwise nonlinear | Multiply upstream by local slope | `log`, `exp`, `tanh`, `power` |
| Matrix multiply | Use transposes to repair shapes | `h @ W2`, `embcat @ W1` |
| View/reshape | Reshape gradient back to original shape | `emb.view(...)` |
| Max | Route gradient to the winning entry | `logits.max(1)` |

The most useful debugging habit:

```text
Every gradient must have the same shape as the tensor it belongs to.
```

When stuck, write the shape first. Usually the correct sum, broadcast, or transpose becomes obvious.
